# Plugging in a ZMART-driver

Write a driver for a pretend microscope, check it, and install it. Run the cells from top to bottom. The [README](README.md) says what every method must hand back.

A driver is a folder with two files. The settings first: the pretend microscope needs nothing to be reached.

In [ ]:
from pathlib import Path

Path("pretend_scope").mkdir(exist_ok=True)

In [ ]:
%%writefile pretend_scope/zmart_driver.json
{
  "name": "pretend-scope",
  "driver": "zmart_driver.py",
  "connection": {
    "microscope": "pretend-01"
  }
}

Then the class, one method per command. Each hands back `True` and the values, or `False` and a message.

In [ ]:
%%writefile pretend_scope/zmart_driver.py
import json
from pathlib import Path

TRAVEL = {"x": [-1000.0, 1000.0], "y": [-1000.0, 1000.0], "z": [-100.0, 100.0]}   # micrometres from the origin
HALF_FIELD = 50.0                                                                   # half a field of view


class ZmartDriver:
    def __init__(self, connection):
        self.position = {"x": 0.0, "y": 0.0, "z": 0.0}
        self.settings = {"exposure_ms": 10.0}
        self.output_root = Path("pretend_scope_images")
        self.output_root.mkdir(exist_ok=True)

    def disconnect(self):
        pass

    def get_info(self):
        return True, "A pretend microscope in memory. exposure_ms is the exposure time in milliseconds. +z points up."

    def get_actuators(self):
        return True, (["motor"], ["motor"], ["motor"])

    def get_xyz(self, with_actuators):
        p = self.position
        canvas = (TRAVEL["x"][0] - HALF_FIELD, TRAVEL["x"][1] + HALF_FIELD,
                  TRAVEL["y"][0] - HALF_FIELD, TRAVEL["y"][1] + HALF_FIELD,
                  TRAVEL["z"][0], TRAVEL["z"][1])
        return True, (p["x"], p["y"], p["z"], "motor", "motor", "motor", canvas)

    def set_xyz(self, x, y, z, with_actuators):
        for axis, value in {"x": x, "y": y, "z": z}.items():
            low, high = TRAVEL[axis]
            if not low <= value <= high:
                return False, f"{axis} = {value} is outside the travel, {low} to {high}"
        self.position = {"x": x, "y": y, "z": z}
        return True, ("motor", "motor", "motor")

    def get_state(self):
        return True, (dict(self.settings), {"objective": "10x"})

    def set_state(self, changeable):
        for name in changeable:
            if name not in self.settings:
                return False, f"unknown setting {name}"
        self.settings.update(changeable)
        return True, dict(changeable)

    def get_acquisition_settings(self):
        return True, {"format": {"options": ["json"], "active": "json"}}

    def acquire(self, position_label, acquisition_settings):
        path = self.output_root / f"{position_label}.json"
        path.write_text(json.dumps({"position": self.position, "settings": self.settings}))
        p = self.position
        plane = {"path": str(path), "c": 0, "z": 0, "t": 0, "x_um": p["x"], "y_um": p["y"], "z_um": p["z"]}
        return True, ([str(path)], [plane])

    def get_procedures(self):
        return True, {}

    def run_procedure(self, procedure):
        return False, f"unknown procedure {procedure.get('name')}"

Check that it fits. An empty list means every method hands back the right thing.

In [ ]:
from zmart_controller import mic

mic.validate_driver("pretend_scope/zmart_driver.json")

Install it, and connect by name.

In [ ]:
mic.register_driver("pretend_scope/zmart_driver.json")
mic.get_instruments()

In [ ]:
mic.connect("pretend-scope")
mic.set_xyz(100, 50, 0)
mic.acquire(position_label="A1")["content"]["files"]

In [ ]:
mic.set_xyz(5000, 0, 0)

In [ ]:
mic.disconnect()

This driver does not need to stay installed.

In [ ]:
mic.remove_driver("pretend-scope")
mic.get_instruments()

---

MIT license. Thom de Hoog, Center for Microscopy and Image Analysis (ZMB), University of Zurich. thom.dehoog@zmb.uzh.ch, thomdehoog@gmail.com.